# Train and evaluate a calibrated ModernBERT LLM router

ModernBERT predicts **whether a faster candidate preserves the quality of a strong fallback**. A deterministic analytical model—not ModernBERT and not candidate inference—estimates latency from model facts and prompt size.

The default run is a **random-split feasibility test**: can prompt content predict safe replacement at all? Repeated prompt content is grouped by a normalized SHA-256 hash, so the same question cannot cross train, validation, and test under different benchmark IDs. After random feasibility succeeds across several seeds, rerun with `SPLIT_MODE = "dataset_ood"` as a separate generalization stress test.

A successful sealed-test POC must activate the router, retain at least 98% quality at a one-sided 95% lower confidence bound, produce positive net analytical latency savings after router overhead, and route at least some prompts away from fallback. Validation uses an additional 1 percentage-point safety margin: it must reach a 99% LCB before the 98% sealed-test gate is opened.

> This proves feasibility under explicit analytical assumptions. It does not claim measured production latency.

## 1. One-cell Google Colab setup

Open this notebook in Google Colab, select a GPU runtime, and run every cell in order. This cell clones the `develop` branch, installs the project normally (not as an editable package), registers `src` in the live kernel, and verifies the import immediately. No terminal, runtime restart, or separate setup notebook is required.

In [1]:
%cd /content
!test -d /content/LLM_Router || git clone --branch develop https://github.com/BrunoVitti96/LLM-router.git /content/LLM_Router
!git -C /content/LLM_Router pull --ff-only origin develop
%cd /content/LLM_Router
%pip install -q -U ".[notebook]"

import importlib
import sys
from pathlib import Path

PROJECT_ROOT = Path("/content/LLM_Router")
SOURCE_ROOT = str(PROJECT_ROOT / "src")
if SOURCE_ROOT not in sys.path:
    sys.path.insert(0, SOURCE_ROOT)
for module_name in list(sys.modules):
    if module_name == "llm_router" or module_name.startswith("llm_router."):
        del sys.modules[module_name]
importlib.invalidate_caches()
import llm_router

print(f"Router package ready from {Path(llm_router.__file__).resolve()}")

/content
Cloning into '/content/LLM_Router'...
remote: Enumerating objects: 176, done.
remote: Counting objects: 100% (176/176), done.
remote: Compressing objects: 100% (115/115), done.
remote: Total 176 (delta 90), reused 140 (delta 54), pack-reused 0 (from 0)
Receiving objects: 100% (176/176), 403.87 KiB | 23.76 MiB/s, done.
Resolving deltas: 100% (90/90), done.
From https://github.com/BrunoVitti96/LLM-router
 * branch            develop    -> FETCH_HEAD
Already up to date.
/content/LLM_Router
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.9/504.9 kB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 117.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 2. Experiment controls and automatic benchmark download

Start with `random`. Five stratified content-group folds produce an approximate 60/20/20 split while keeping repeated prompts together. Use `dataset_ood` only for a separate, harder artifact. For a serious conclusion, repeat both modes with at least seeds 42, 43, and 44. Never tune a later run from an already-opened test result.

The official archive is about 1.28 GB. The loader discovers its `dataset/split/model/file.json` structure instead of assuming a fixed wrapper name. No candidate LLM is installed or executed.

In [2]:
import shutil
import tarfile
from dataclasses import replace
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from huggingface_hub import hf_hub_download
from IPython.display import display

from llm_router.config import DEFAULT_CONFIG
from llm_router.modernbert_poc import (
    export_modernbert_hybrid_poc,
    train_modernbert_hybrid_poc,
)
from llm_router.oracle import oracle_choices
from llm_router.public_benchmark import (
    EconomicsScenario,
    ModelProfile,
    benchmark_inventory,
    export_public_benchmark,
    load_llmrouterbench,
    make_complete_panel,
    run_public_benchmark,
    simulate_economics,
    split_benchmark,
)
from llm_router.utils.training import seed_everything

SEED = 42
SPLIT_MODE = "random"  # Use "dataset_ood" only for a separate stress test.
EPOCHS = 8  # Maximum; validation early stopping usually finishes sooner.
MINIMUM_EPOCHS = 2
EARLY_STOPPING_PATIENCE = 2
VALIDATION_QUALITY_MARGIN = 0.01
ROUTER_CONFIG = replace(DEFAULT_CONFIG, seed=SEED)
DATA_ROOT = Path("/content/LLMRouterBench")
OUTPUT_DIR = Path(
    f"reports_benchmark/modernbert_hybrid_{SPLIT_MODE}_seed_{SEED}"
)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

seed_everything(SEED)
inventory = benchmark_inventory(DATA_ROOT)
archive_preview = []
if inventory.empty:
    archive = hf_hub_download(
        repo_id="NPULH/LLMRouterBench",
        filename="bench-release.tar.gz",
        repo_type="dataset",
    )
    results_dir = DATA_ROOT / "results"
    results_dir.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive, "r:gz") as bundle:
        archive_preview = bundle.getnames()[:12]
        bundle.extractall(results_dir, filter="data")
    inventory = benchmark_inventory(DATA_ROOT)
if inventory.empty:
    extracted_preview = [
        str(path.relative_to(DATA_ROOT))
        for path in DATA_ROOT.rglob("*")
        if path.is_file()
    ][:20]
    raise RuntimeError(
        "No dataset/split/model/*.json benchmark layout was found after "
        f"extraction. Archive entries: {archive_preview}. "
        f"Extracted files: {extracted_preview}"
    )
DATA_ROOT = Path(inventory.iloc[0]["file"]).parents[3]
print({
        "device": DEVICE,
        "split_mode": SPLIT_MODE,
        "seed": SEED,
        "data_root": str(DATA_ROOT.resolve()),
        "output_dir": str(OUTPUT_DIR),
    })
if DEVICE == "cpu":
    print("Warning: CPU training works for a smoke test but will be slow. A GPU is recommended.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


bench-release.tar.gz:   0%|          | 0.00/1.28G [00:00<?, ?B/s]

{'device': 'cuda', 'split_mode': 'random', 'seed': 42, 'data_root': '/content/LLMRouterBench/results/bench-release', 'output_dir': 'reports_benchmark/modernbert_hybrid_random_seed_42'}


## 3. Inspect available datasets and models

Do this before editing model profiles. The strings in `SELECTED_MODELS` must exactly match model directory names shown below. Choose at least two candidates and at least three datasets for a dataset-disjoint split.

In [3]:
inventory = benchmark_inventory(DATA_ROOT)
assert not inventory.empty, "No LLMRouterBench result files were found."
inventory_summary = (
    inventory.groupby(["model", "dataset", "source_split"])
    .size()
    .rename("files")
    .reset_index()
)
display(inventory_summary)
print(f"Available models: {inventory.model.nunique()}")
print(f"Available datasets: {inventory.dataset.nunique()}")

,model,dataset,source_split,files
0,DeepHermes-3-Llama-3-8B-Preview,aime,hybrid,1
1,DeepHermes-3-Llama-3-8B-Preview,arcc,test,1
2,DeepHermes-3-Llama-3-8B-Preview,bbh,test,1
3,DeepHermes-3-Llama-3-8B-Preview,emorynlp,test,1
4,DeepHermes-3-Llama-3-8B-Preview,finqa,test,1
...,...,...,...,...
562,qwen3-235b-a22b-thinking-2507,mmlupro,test_1000,1
563,qwen3-235b-a22b-thinking-2507,mmlupro,test_3000,1
564,qwen3-235b-a22b-thinking-2507,simpleqa,subset_500,1
565,qwen3-235b-a22b-thinking-2507,simpleqa,test,1


Available models: 40
Available datasets: 24


## 4. Declare a non-dominated candidate panel

The previous run included NVIDIA-Nemotron-Nano-9B-v2, but it was analytically slower than the Qwen fallback and had a 0% oracle-selection rate. It could never change the latency-minimizing decision.

The clean default is therefore:

- `Fin-R1`: approximately 7B parameters, the faster replacement candidate;
- `Qwen3-8B`: 8.2B parameters, the potential quality fallback.

Both generate autoregressively at BF16 precision. The public pool contains no diffusion model; add one only when matching pre-collected quality results and sourced denoising settings exist.

In [4]:
MODEL_PROFILES = {
    "Fin-R1": {
        "parameters_billions": 7.0,
        "active_parameters_billions": 7.0,
        "architecture": "autoregressive",
        "weight_bits": 16,
    },
    "Qwen3-8B": {
        "parameters_billions": 8.2,
        "active_parameters_billions": 8.2,
        "architecture": "autoregressive",
        "weight_bits": 16,
    },
}
SELECTED_MODELS = tuple(MODEL_PROFILES)
assert len(SELECTED_MODELS) >= 2, "Routing requires at least two models."
missing_models = sorted(set(SELECTED_MODELS) - set(inventory.model))
assert not missing_models, (
    f"Configured models were not found: {missing_models}. "
    f"Available models: {sorted(inventory.model.unique())}"
)
print(SELECTED_MODELS)

('Fin-R1', 'Qwen3-8B')


## 5. Load pre-collected quality and calculate analytical latency

The benchmark provides candidate answers and quality scores. `simulate_economics` calculates latency without loading those candidates. Expected output length is a bounded function of prompt length, so the policy cannot peek at a candidate's realized response length. The panel also records a normalized prompt hash and duplicate-group size for content-level leakage auditing.

In [5]:
profiles = tuple(
    ModelProfile(
        name=name,
        input_price_per_million=0.0,
        output_price_per_million=0.0,
        **settings,
    )
    for name, settings in MODEL_PROFILES.items()
)
scenario = EconomicsScenario(
    name="notebook-analytical-latency-poc",
    as_of=pd.Timestamp.utcnow().date().isoformat(),
    profiles=profiles,
    latency_method="analytical",
    effective_tflops=60.0,
    memory_bandwidth_gbps=900.0,
    fixed_model_overhead_s=0.015,
    router_overhead_s=0.004,
    output_base_tokens=24.0,
    output_tokens_per_prompt_token=0.20,
    output_min_tokens=16,
    output_max_tokens=256,
    notes="POC assumptions; not measured production latency.",
)

records = load_llmrouterbench(DATA_ROOT, models=SELECTED_MODELS)
simulated = simulate_economics(records, scenario)
panel = make_complete_panel(simulated, models=SELECTED_MODELS)
assert simulated.latency_source.eq("analytical").all()
print({"complete_prompts": len(panel.examples), "models": panel.models})
print({
    "repeated_prompt_rows": int((panel.examples.prompt_group_size > 1).sum()),
    "largest_prompt_group": int(panel.examples.prompt_group_size.max()),
})
display(
    simulated.groupby("model")["simulated_latency_s"]
    .agg(["min", "median", "mean", "max"])
    .sort_values("mean")
)

{'complete_prompts': 14041, 'models': ('Fin-R1', 'Qwen3-8B')}
{'repeated_prompt_rows': 135, 'largest_prompt_group': 4}


,min,median,mean,max
model,,,,
Fin-R1,0.662578,1.277956,1.707854,4.870356
Qwen3-8B,0.691318,1.412189,1.920396,5.696962


### Leakage check

This deliberately changes every realized completion length. Analytical latency must remain identical because only prompt size and declared model/scenario properties are allowed to affect it.

In [6]:
counterfactual_records = records.copy()
counterfactual_records["completion_tokens"] *= 100
counterfactual = simulate_economics(counterfactual_records, scenario)
assert np.allclose(
    simulated.simulated_latency_s,
    counterfactual.simulated_latency_s,
), "Analytical latency unexpectedly depends on realized completion length."
print("Passed: candidate completion tokens do not affect analytical latency.")

Passed: candidate completion tokens do not affect analytical latency.


## 6. Freeze train, validation, and sealed-test splits

The fallback is selected from training quality only. Validation is used for checkpoint selection, out-of-fold calibration, threshold selection, and the activation gate. Test outcomes remain sealed until the complete policy is frozen.

- `random`: normalized-prompt-hash-disjoint feasibility test; start here.
- `dataset_ood`: dataset-disjoint generalization stress test; run separately.

In [7]:
split = split_benchmark(panel, mode=SPLIT_MODE, seed=SEED)
split_summary = pd.DataFrame(
    {
        "split": ["train", "validation", "test"],
        "prompts": [len(split.train), len(split.validation), len(split.test)],
        "datasets": [
            split.train_datasets,
            split.validation_datasets,
            split.test_datasets,
        ],
    }
)
display(split_summary)
assert not set(split.train).intersection(split.validation)
assert not set(split.train).intersection(split.test)
assert not set(split.validation).intersection(split.test)
split_hashes = {
    name: set(panel.examples.iloc[indices].prompt_hash)
    for name, indices in {
        "train": split.train,
        "validation": split.validation,
        "test": split.test,
    }.items()
}
assert split_hashes["train"].isdisjoint(split_hashes["validation"])
assert split_hashes["train"].isdisjoint(split_hashes["test"])
assert split_hashes["validation"].isdisjoint(split_hashes["test"])
if SPLIT_MODE == "dataset_ood":
    assert set(split.train_datasets).isdisjoint(split.validation_datasets)
    assert set(split.train_datasets).isdisjoint(split.test_datasets)
    assert set(split.validation_datasets).isdisjoint(split.test_datasets)

,split,prompts,datasets
0,train,8425,"(aime, arcc, bbh, emorynlp, finqa, gpqa, human..."
1,validation,2804,"(aime, arcc, bbh, emorynlp, finqa, gpqa, human..."
2,test,2812,"(aime, arcc, bbh, emorynlp, finqa, gpqa, human..."


## 7. Verify validation-only oracle headroom and scenario sensitivity

The hindsight oracle sees recorded outcomes and chooses the fastest model that matches or beats fallback quality. It is unattainable at deployment, but it proves whether routing opportunity exists.

This cell uses **validation only** and repeats the calculation under conservative compute, conservative bandwidth, longer-output, and larger fixed-overhead assumptions. If modest changes erase headroom, the latency claim is too fragile to train. Router-overhead sensitivity is evaluated later on the frozen test policy because it depends on how often the router selects the faster model.

In [8]:
def validation_oracle_metrics(candidate_panel):
    fallback_index = int(candidate_panel.score[split.train].mean(axis=0).argmax())
    target = oracle_choices(
        candidate_panel.score,
        candidate_panel.latency,
        fallback_index=fallback_index,
    )
    indices = split.validation
    rows = np.arange(len(indices))
    chosen = target[indices]
    fallback_quality = candidate_panel.score[indices, fallback_index].mean()
    oracle_quality = candidate_panel.score[indices][rows, chosen].mean()
    fallback_latency = candidate_panel.latency[indices, fallback_index].mean()
    oracle_latency = candidate_panel.latency[indices][rows, chosen].mean()
    return {
        "fallback_model": candidate_panel.models[fallback_index],
        "quality_retention": oracle_quality / max(fallback_quality, 1e-12),
        "latency_savings": 1 - oracle_latency / fallback_latency,
        "fallback_usage": np.mean(chosen == fallback_index),
    }

sensitivity_settings = {
    "balanced": {},
    "compute_conservative": {"effective_tflops": 40.0},
    "bandwidth_conservative": {"memory_bandwidth_gbps": 600.0},
    "larger_fixed_overhead": {"fixed_model_overhead_s": 0.050},
    "longer_outputs": {
        "output_base_tokens": 48.0,
        "output_tokens_per_prompt_token": 0.35,
    },
}
sensitivity_rows = []
for name, overrides in sensitivity_settings.items():
    variant = replace(scenario, name=name, **overrides)
    variant_records = simulate_economics(records, variant)
    variant_panel = make_complete_panel(variant_records, models=SELECTED_MODELS)
    assert variant_panel.examples.example_id.equals(panel.examples.example_id)
    sensitivity_rows.append(
        {"scenario": name, **validation_oracle_metrics(variant_panel)}
    )
sensitivity = pd.DataFrame(sensitivity_rows)
display(sensitivity)
assert (sensitivity.latency_savings > 0).all(), (
    "Oracle headroom is not robust across the declared sensitivity scenarios."
)

,scenario,fallback_model,quality_retention,latency_savings,fallback_usage
0,balanced,Qwen3-8B,1.091599,0.084319,0.242154
1,compute_conservative,Qwen3-8B,1.091599,0.084130,0.242154
2,bandwidth_conservative,Qwen3-8B,1.091599,0.084671,0.242154
3,larger_fixed_overhead,Qwen3-8B,1.091599,0.082834,0.242154
4,longer_outputs,Qwen3-8B,1.091599,0.083940,0.242154


## 8. Understand the objective, loss, and calibration

For each non-fallback candidate, replacement safety is:

$$y_m(x)=\mathbf 1[Q_m(x)\ge Q_f(x)-\epsilon_q].$$

The deployable head now uses **class-balanced** binary cross-entropy. Candidate $m$ receives positive weight $N_{unsafe}/N_{safe}$, clipped for numerical stability. This prevents a high safe-rate prior from winning without learning prompt differences.

A second, training-only head imitates the hindsight oracle and penalizes expected quality risk plus latency regret:

$$\mathcal L_{oracle}=(1+g_o)CE(z,o)+4\sum_m p_m d_m+\sum_m p_m r_m.$$

The complete objective is:

$$\boxed{\mathcal L_{train}=\mathcal L_{balanced\ safety}+0.25\mathcal L_{oracle}}.$$

After checkpoint selection, each safety logit receives per-candidate Platt calibration. Threshold search uses **out-of-fold validation probabilities**, so a validation row never calibrates itself. The exported deployment scaler is fitted on all validation rows. ModernBERT never predicts latency or the final model.

## 9. Train ModernBERT

Exact dataset names are excluded from router inputs. ModernBERT sees prompt text and prompt-token count, reducing brittle dataset-identity memorization. The LoRA adapter uses a lower learning rate than the newly initialized heads, and validation early stopping prevents the falling training loss from hiding overfitting. On FP16 GPUs, skipped optimizer steps do not advance the learning-rate scheduler.

The input diagnostics use ModernBERT's own tokenizer. For example, if `truncation_rate=0.12`, then 12% of prompts exceeded the 512-token router limit; that would motivate a longer limit or chunked encoding in a later experiment.

In [9]:
training = train_modernbert_hybrid_poc(
    panel,
    split,
    config=ROUTER_CONFIG,
    epochs=EPOCHS,
    batch_size=8,
    learning_rate=1e-4,
    head_learning_rate=2e-4,
    minimum_epochs=MINIMUM_EPOCHS,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    quality_epsilon=0.0,
    safety_loss_weight=1.0,
    oracle_auxiliary_weight=0.25,
    device=DEVICE,
)
display(training.history)
display(training.calibration_diagnostics)
display(pd.DataFrame([training.input_diagnostics]))
print({
    "best_epoch": training.best_epoch,
    "epochs_completed": training.epochs_completed,
    "stopped_early": training.stopped_early,
    "training_seconds": round(training.training_seconds, 1),
})
assert training.safety_probabilities.shape == panel.score.shape
assert np.allclose(
    training.safety_probabilities[:, training.fallback_index], 1.0
)
assert np.all(
    (training.safety_probabilities >= 0)
    & (training.safety_probabilities <= 1)
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/596M [00:00<?, ?B/s]

W0818 20:58:23.508000 548 torch/_inductor/utils.py:1731] [1/0_1] Not enough SMs to use max_autotune_gemm mode


,epoch,train_total_loss,train_safety_loss,train_oracle_auxiliary_loss,validation_total_loss,validation_safety_loss,validation_oracle_auxiliary_loss,skipped_optimizer_steps,encoder_learning_rate,head_learning_rate
0,1,0.572555,0.298712,1.095369,0.583830,0.305811,1.112076,2,0.000098,0.000197
1,2,0.531234,0.278240,1.011975,0.569227,0.297130,1.088386,2,0.000089,0.000179
2,3,0.512748,0.269132,0.974463,0.564344,0.294930,1.077654,3,0.000074,0.000148
3,4,0.495843,0.260681,0.940648,0.583219,0.308469,1.098999,3,0.000054,0.000108
4,5,0.474497,0.249864,0.898535,0.571653,0.299804,1.087396,4,0.000034,0.000068


,candidate,validation_examples,safe_prevalence,raw_brier,calibrated_brier,raw_ece,calibrated_ece
0,Fin-R1,2804,0.757846,0.177067,0.150904,0.128773,0.017545


,examples,max_input_tokens,truncated_examples,truncation_rate,router_tokens_p50,router_tokens_p95,router_tokens_max
0,14041,512,474,0.033758,85.0,463.0,1408


{'best_epoch': 3, 'epochs_completed': 5, 'stopped_early': True, 'training_seconds': 1940.3}


## 10. Freeze the policy, then open the sealed test

Validation selects the confidence threshold from a dense, predeclared grid and may disable the router. Selection requires `98% + 1% margin = 99%` validation LCB; the sealed-test gate remains 98%. This buffer reduces the chance that an exactly-on-the-line validation result fails after deployment. Only after that policy is frozen does the report evaluate test outcomes. `poc_passed` is stricter than `router_active`: the sealed test must also pass the quality, savings, and non-trivial-routing gates.

In [10]:
result = run_public_benchmark(
    panel,
    split,
    objective="latency",
    minimum_quality_retention=0.98,
    confidence=0.95,
    validation_quality_margin=VALIDATION_QUALITY_MARGIN,
    minimum_predicted_savings=0.02,
    router_overhead_s=scenario.router_overhead_s,
    seed=SEED,
    routing_probabilities=training.safety_probabilities,
    router_name="modernbert_hybrid_router",
)
assert result.fallback_model == panel.models[training.fallback_index]
display(result.threshold_search)
display(result.summary)
display(result.candidate_diagnostics)
router_dataset_metrics = result.per_dataset_metrics.loc[
    result.per_dataset_metrics.strategy.eq("modernbert_hybrid_router")
].sort_values("quality_retention")
display(router_dataset_metrics)
overhead_sensitivity = result.router_overhead_sensitivity
display(overhead_sensitivity)
print({
    "break_even_router_overhead_ms": round(
        overhead_sensitivity.break_even_router_overhead_ms.iloc[0], 2
    )
})
print(
    {
        "poc_passed": result.poc_passed,
        "router_active": result.router_active,
        "selected_threshold": result.selected_threshold,
        "fallback_model": result.fallback_model,
        "failure_reasons": result.failure_reasons,
    }
)

,threshold,quality,fallback_quality,quality_retention,quality_retention_lcb,quality_loss_rate,quality_loss_rate_ucl,routed_safety_precision,routed_fraction,macro_dataset_quality_retention,macro_dataset_quality_retention_lcb,macro_dataset_count,mean_resource,fallback_resource,resource_savings,fallback_usage
0,0.500,0.607703,0.704708,0.862348,0.842029,0.159415,0.171114,0.817402,0.873039,0.857711,0.781109,18.0,1.769228,1.952785,0.093998,0.126961
1,0.600,0.630171,0.704708,0.894231,0.875002,0.135164,0.146137,0.835646,0.822397,0.908219,0.848705,18.0,1.783578,1.952785,0.086649,0.177603
2,0.700,0.650143,0.704708,0.922571,0.904586,0.111983,0.122155,0.851185,0.752496,0.947650,0.900009,18.0,1.800206,1.952785,0.078134,0.247504
3,0.750,0.661198,0.704708,0.938259,0.921183,0.097718,0.107333,0.857736,0.686876,0.966113,0.922169,18.0,1.816073,1.952785,0.070009,0.313124
4,0.800,0.675820,0.704708,0.959008,0.943285,0.078459,0.087224,0.866424,0.587375,0.984807,0.944514,18.0,1.833105,1.952785,0.061287,0.412625
5,0.850,0.686519,0.704708,0.974190,0.961024,0.053852,0.061303,0.871925,0.420471,0.989138,0.956333,18.0,1.866162,1.952785,0.044359,0.579529
6,0.855,0.688302,0.704708,0.976721,0.963843,0.050999,0.058275,0.871518,0.396933,0.991896,0.959220,18.0,1.870139,1.952785,0.042322,0.603067
7,0.860,0.690086,0.704708,0.979251,0.967117,0.045292,0.052202,0.875855,0.364836,0.989658,0.960039,18.0,1.877145,1.952785,0.038734,0.635164
8,0.865,0.693652,0.704708,0.984312,0.972876,0.039230,0.045718,0.882854,0.334879,0.993994,0.965258,18.0,1.883125,1.952785,0.035672,0.665121
9,0.870,0.694009,0.704708,0.984818,0.973909,0.036020,0.042270,0.881871,0.304922,0.993315,0.964974,18.0,1.888695,1.952785,0.032820,0.695078


,objective,selected_model,quality,fallback_quality,quality_retention,quality_retention_lcb,quality_loss_rate,quality_loss_rate_ucl,routed_safety_precision,routed_fraction,macro_dataset_quality_retention,macro_dataset_quality_retention_lcb,macro_dataset_count,mean_resource,fallback_resource,resource_savings,fallback_usage,oracle_savings_capture
best_single,latency,Qwen3-8B,0.713371,0.713371,1.000000,1.000000,0.000000,0.000961,1.000000,0.000000,1.000000,1.000000,18.0,1.884562,1.884562,0.000000,1.000000,0.000000
cheapest_single,latency,Fin-R1,0.548720,0.713371,0.769192,0.746394,0.233286,0.246657,0.766714,1.000000,0.749941,0.613272,18.0,1.677601,1.884562,0.109819,0.000000,1.337521
dataset_lookup,latency,dynamic,0.725462,0.713371,1.016949,1.009633,0.008179,0.011484,0.900862,0.082504,1.037588,0.992888,18.0,1.877295,1.884562,0.003856,0.917496,0.046960
outcome_oracle,latency,dynamic,0.782006,0.713371,1.096211,1.085216,0.000000,0.000961,1.000000,0.766714,1.120543,1.064477,18.0,1.729827,1.884562,0.082107,0.233286,1.000000
modernbert_hybrid_router,latency,dynamic,0.715505,0.713371,1.002991,0.994709,0.017070,0.021577,0.924290,0.225462,1.002785,0.984388,18.0,1.841561,1.884562,0.022817,0.774538,0.277898


,split,model,is_fallback,mean_quality,mean_resource,relative_resource_vs_fallback,safe_rate,candidate_better_rate,candidate_equal_rate,candidate_worse_rate,both_zero_rate,faster_than_fallback_rate,oracle_selection_rate
0,train,Fin-R1,False,0.541484,1.708869,0.889306,0.768427,0.062315,0.706113,0.231573,0.226944,1.0,0.768427
1,train,Qwen3-8B,True,0.710742,1.921576,1.000000,1.000000,0.000000,1.000000,0.000000,0.289258,0.0,0.231573
2,validation,Fin-R1,False,0.527104,1.735142,0.888547,0.757846,0.064551,0.693295,0.242154,0.230742,1.0,0.757846
3,validation,Qwen3-8B,True,0.704708,1.952785,1.000000,1.000000,0.000000,1.000000,0.000000,0.295292,0.0,0.242154
4,test,Fin-R1,False,0.548720,1.677601,0.890181,0.766714,0.068634,0.698080,0.233286,0.217994,1.0,0.766714
5,test,Qwen3-8B,True,0.713371,1.884562,1.000000,1.000000,0.000000,1.000000,0.000000,0.286629,0.0,0.233286


,strategy,dataset,prompts,quality,fallback_quality,quality_retention,quality_retention_lcb,quality_loss_rate,quality_loss_rate_ucl,routed_safety_precision,routed_fraction,macro_dataset_quality_retention,macro_dataset_quality_retention_lcb,macro_dataset_count,mean_resource,fallback_resource,resource_savings,fallback_usage
77,modernbert_hybrid_router,gpqa,39,0.717949,0.769231,0.933333,0.856821,0.051282,0.143665,0.333333,0.076923,0.933333,0.933333,1.0,1.329245,1.329514,0.000202,0.923077
76,modernbert_hybrid_router,finqa,223,0.699552,0.721973,0.968944,0.926865,0.049327,0.079046,0.902655,0.506726,0.968944,0.968944,1.0,4.038165,4.338614,0.069250,0.493274
73,modernbert_hybrid_router,arcc,259,0.918919,0.942085,0.975410,0.950263,0.038610,0.063542,0.952153,0.806950,0.975410,0.975410,1.0,0.859082,0.900231,0.045709,0.193050
87,modernbert_hybrid_router,meld,242,0.533058,0.541322,0.984733,0.967013,0.008264,0.024664,0.818182,0.045455,0.984733,0.984733,1.0,1.411230,1.411865,0.000450,0.954545
89,modernbert_hybrid_router,winogrande,239,0.853556,0.861925,0.990291,0.970719,0.016736,0.036756,0.862069,0.121339,0.990291,0.990291,1.0,0.935702,0.939515,0.004058,0.878661
78,modernbert_hybrid_router,humaneval,31,0.580645,0.580645,1.000000,1.000000,0.000000,0.080270,1.000000,0.290323,1.000000,1.000000,1.0,1.711860,1.755690,0.024964,0.709677
79,modernbert_hybrid_router,kandk,138,0.768116,0.768116,1.000000,1.000000,0.000000,0.019228,1.000000,0.000000,1.000000,1.000000,1.0,1.340871,1.336871,-0.002992,1.000000
75,modernbert_hybrid_router,emorynlp,152,0.296053,0.296053,1.000000,1.000000,0.000000,0.017488,1.000000,0.000000,1.000000,1.000000,1.0,1.847711,1.843711,-0.002170,1.000000
86,modernbert_hybrid_router,medqa,266,0.800752,0.800752,1.000000,1.000000,0.000000,0.010069,1.000000,0.007519,1.000000,1.000000,1.0,1.514361,1.510943,-0.002262,0.992481
81,modernbert_hybrid_router,livecodebench,222,0.684685,0.684685,1.000000,1.000000,0.000000,0.012040,1.000000,0.000000,1.000000,1.000000,1.0,2.656569,2.652569,-0.001508,1.000000


,router_overhead_ms,net_latency_savings,break_even_router_overhead_ms
0,4.0,0.022817,47.000473
1,10.0,0.019633,47.000473
2,20.0,0.014327,47.000473
3,50.0,-0.001592,47.000473


{'break_even_router_overhead_ms': np.float64(47.0)}
{'poc_passed': True, 'router_active': True, 'selected_threshold': 0.885, 'fallback_model': 'Qwen3-8B', 'failure_reasons': ()}


### Interpret the result honestly

- **Oracle savings near zero:** the panel or assumptions offer no useful headroom.
- **Oracle headroom but `router_active=False`:** ModernBERT found no validation-safe policy.
- **Router active but `poc_passed=False`:** validation looked promising, but sealed-test behavior did not generalize.
- **Random passes and dataset-OOD fails:** feasibility exists, but cross-domain generalization does not.
- **Both modes pass across several seeds and sensitivity scenarios:** this is a credible analytical-latency POC.

A fallback-only outcome proves the safety guard worked; it does not prove the router worked.

## 11. Export the complete report and router artifact

The export contains strategy metrics, threshold search, sealed-test decisions, candidate and per-dataset diagnostics, harm-rate confidence bounds, analytical and router-overhead sensitivity, input-truncation diagnostics, explicit pass/fail reasons, training history, calibration diagnostics, Platt parameters, LoRA weights, both heads, and tokenizer.

Numerical example: a 2% pooled saving can still be unacceptable if one small dataset retains only 90% quality, or if a measured 50 ms router overhead exceeds the reported break-even overhead. The additional CSVs make both problems visible.

In [11]:
report_dir = export_public_benchmark(result, scenario, OUTPUT_DIR)
sensitivity.to_csv(report_dir / "validation_sensitivity.csv", index=False)
artifact_dir = export_modernbert_hybrid_poc(
    training,
    panel.models,
    report_dir / "modernbert_router",
    selected_threshold=result.selected_threshold,
    router_active=result.router_active,
    poc_passed=result.poc_passed,
    failure_reasons=result.failure_reasons,
    minimum_predicted_savings=0.02,
    validation_quality_margin=VALIDATION_QUALITY_MARGIN,
    config=ROUTER_CONFIG,
)
print({"reports": str(report_dir.resolve()), "artifact": str(artifact_dir.resolve())})

{'reports': '/content/LLM_Router/reports_benchmark/modernbert_hybrid_random_seed_42', 'artifact': '/content/LLM_Router/reports_benchmark/modernbert_hybrid_random_seed_42/modernbert_router'}


## 12. Download the result before Colab shuts down

The ZIP path contains split mode and seed, so feasibility and dataset-OOD artifacts cannot silently overwrite one another.

In [12]:
bundle_path = shutil.make_archive(
    str(OUTPUT_DIR.resolve()), "zip", root_dir=OUTPUT_DIR
)
print(f"Created {bundle_path}")
try:
    from google.colab import files

    files.download(bundle_path)
except ImportError:
    print("Not running in Colab; download the ZIP from the printed path.")

Created /content/LLM_Router/reports_benchmark/modernbert_hybrid_random_seed_42.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 13. Final POC checklist

Before presenting a result, confirm:

- random feasibility passed across at least three seeds;
- dataset-OOD was run as a separate, clearly labeled stress-test artifact;
- every retained alternative has a non-zero oracle-selection rate;
- normalized prompt hashes are disjoint across random train, validation, and test;
- calibration improved or did not materially worsen Brier score and ECE;
- per-dataset retention, harm-rate UCL, and selected-route safety precision are acceptable;
- ModernBERT input truncation is reported and investigated if material;
- no candidate inference or realized completion length was used for latency;
- fallback selection used training quality only;
- checkpointing, calibration, threshold selection, and activation used validation only;
- test outcomes were opened only after policy freeze;
- net savings include router overhead and remain positive across plausible overhead values;
- conclusions survive all declared sensitivity scenarios; and
- every latency claim says **analytical** or **simulated**, never measured.